# A3.6 · Human approval that survives volume

**Function A — Securing AI Architectures → Securing the Architecture — Runtime and the Gateway**  ·  *Security of AI*

Builds on **[A3.5 · Validating what comes back](https://spbreed.github.io/cyber-commons/lessons/A3.5.html)**.

| | |
|---|---|
| Tools used | standard library only |

## What this lesson is

**What it covers.** Route actions by reversibility and measure how many reach a human under each policy.

**Why a security engineer needs it.** An approval queue at volume approves everything, and the risk register still records it as a control. The control it builds is: approval reserved for irreversible actions only, with machine-generated content labelled as such.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Approval works for the rare and irreversible and fails for everything else. The design question is not whether to have a human in the loop — it is how few decisions you can put in front of them, so that each one gets read.

> **At CyberTravels.** Approval is right for a $5,000 refund and wrong for a hotel search. The design question for CyberTravels is not whether to have a human in the loop but how few decisions reach them, so each one gets read. R2.

## 2 · The framework

```
   what reaches the human            what does not
   +-------------------------+       +------------------------+
   | irreversible            |       | reversible             |
   | above a value threshold |       | inside a budget        |
   | outside normal pattern  |       | matching prior approval|
   +-------------------------+       +------------------------+
        a few per day                    everything else

   the control is the filter, not the click
```

**Mitigates: T10 Overwhelming Human-in-the-Loop · T15 Human Manipulation.**

A1.15 showed approval collapsing under volume while still reporting 100%
coverage. The fix is not a better reviewer or a nicer queue. It is **sending
fewer things**.

Route by **reversibility**, because that is what a human is actually useful for:

- **Reversible, bounded** — no approval. Policy from A3.1 decides, and the
  action can be undone if it was wrong.
- **Reversible, expensive to undo** — no approval, but recorded prominently and
  sampled after the fact.
- **Irreversible or externally visible** — approval, every time. Sending mail,
  paying, publishing, deleting without a backup, rotating a credential.

The test for whether your gate will hold is arithmetic, not intent: **how many
requests per day reach a human?** If the answer is more than a person can
consider properly, the control is already a click, and the number tells you so
before the incident does.

The T15 half is one line of implementation and easy to skip: **mark
machine-generated content as machine-generated** wherever a human reads it. A
recommendation that arrives with institutional formatting recruits authority it
has not earned. Labelling it does not stop anyone acting on it — it restores the
scepticism they would apply to a colleague.

> **What this control closes.**
>
> Sends **fewer** things to humans, so the ones that arrive are read. The test is arithmetic: how many per day reach a person.

## 3 · Deciding what needs a human, as a skill

Routing by reversibility only works if somebody has computed what each CyberTravels agent can actually reach and damage in one run. That is a blast-radius review, and its output is an autonomy level rather than an opinion: enumerate the reachable actions with attacker-chosen arguments, not the happy path, and include time-to-stop, because how fast you can halt a run is part of how much it can cost. This is the file in this repository:

### The skill — [`skills/architecture/blast-radius-review/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/architecture/blast-radius-review/SKILL.md)

```yaml
name: blast-radius-review
description: >-
  Compute what an agent can reach and damage in a single run, and decide the
  autonomy level its blast radius can support. Use when reviewing an agent
  design or deployment, deciding whether an action needs human approval, sizing
  a sandbox, or answering how bad it would be if an agent were fully
  compromised.
allowed-tools: Read, Grep, Glob
```

# Blast radius as a design metric

Blast radius is not an adjective. It is the set of resources an agent can
change before anyone can stop it, and it is **computed** from three inputs:

```
blast_radius = reachable_resources × action_irreversibility × time_to_human_stop
```

Treating it as a number is what lets it be a design constraint instead of a
discussion.

## When to use this

At design review, before raising an agent's autonomy, and after any change that
adds a tool, a credential, or a scheduled trigger.

## Procedure

**1 — Enumerate reachable resources.** For each tool the agent can call, list
what it can touch with attacker-chosen arguments — not what it touches in the
happy path. A `Bash` tool with unrestricted arguments reaches everything the
process can reach; record it that way rather than as one row.

**2 — Grade irreversibility.** Per action:

| Grade | Meaning | Example |
|---|---|---|
| 0 | read-only | query, list |
| 1 | reversible with effort | write a file, open a PR |
| 2 | reversible only with a backup | delete a row, force-push |
| 3 | irreversible or externally visible | send an email, pay, publish, rotate a key |

Grade 3 actions are the whole reason approval gates exist. An agent whose
worst action is grade 0 does not need one.

**3 — Measure time-to-human-stop.** How long between the agent deciding and a
human being able to intervene? Interactive with a prompt is seconds. A
scheduled run at 03:00 with notifications off is hours. This term dominates the
product more often than people expect, and it is usually the cheapest to fix.

**4 — Place it on the autonomy ladder.**

| Level | Meaning | Requires |
|---|---|---|
| L1 | suggests; human executes | nothing |
| L2 | acts within a bounded sandbox | reversible actions only |
| L2.5 | acts, but grade-3 actions need approval | a working approval path |
| L3 | acts unattended | demonstrated containment + audit + stop authority |

An agent at L3 whose grade-3 actions are unbounded is misclassified, not brave.

**5 — Find the cheapest reduction.** Usually one of: remove a credential from
the environment, split one broad tool into two narrow ones, add a choke point
in front of the irreversible action, or shorten time-to-stop with a
notification. Recommend the one with the best radius reduction per unit of
friction, and say what it costs.

## Output contract

```json
{
  "resources": [{"tool": "str", "reachable": ["str"], "unbounded": false}],
  "actions": [{"action": "str", "irreversibility": 0, "why": "str"}],
  "time_to_human_stop_seconds": 0,
  "blast_radius": {"score": 0, "inputs": {"resources": 0, "max_irreversibility": 0, "seconds": 0}},
  "autonomy": {"current": "L1|L2|L2.5|L3", "supported": "L1|L2|L2.5|L3", "mismatch": false},
  "reductions": [{"change": "str", "new_score": 0, "friction": "low|medium|high"}]
}
```

Show `inputs`. A blast-radius score without its terms cannot be challenged, and
an unchallengeable metric stops being used.

## Failure modes

- **Counting the happy path.** Enumerate with attacker-chosen arguments.
- **Ignoring time-to-stop** because it is not about permissions. It is the term
  that separates an incident from a near miss.
- **Raising autonomy because the agent has been reliable.** Reliability is not
  containment; it is the absence of an adversary so far.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/architecture/blast-radius-review/scripts/blast_radius_review.py
SCRIPT = "skills/architecture/blast-radius-review/scripts/blast_radius_review.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

The skill loads and reports its shape. The failure mode to carry into your own estate is the last one: raising an agent's autonomy because it has been reliable. Reliability is a measurement of the happy path; blast radius is a measurement of the worst one, and only the second bounds what an approval gate is for.

## Your turn

Count how many approvals your agents generate daily and compare it with 25. If you are above it, decide which actions are reversible enough to be handled by policy instead — that list is usually most of them.

---

**Next → [A3.7 · The agent gateway: one choke point when you scale](https://spbreed.github.io/cyber-commons/lessons/A3.7.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A3.6.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A3.6.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*